# GestuRhythm - Gesture Transformer Training
**Input:** sequences.npy (719, 30, 126) + labels.npy (719, 6)
**Output:** gesture_transformer.pt

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

X = np.load('/kaggle/input/gesturhythm/sequences.npy')
y = np.load('/kaggle/input/gesturhythm/labels.npy')
print(f'X: {X.shape} | y: {y.shape}')

## 1. Phan tich Dataset

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('GestuRhythm - Dataset Analysis', fontsize=14, fontweight='bold')

# 1. Phan bo Pitch
pitch_map = {60:'C4', 62:'D4', 65:'F4', 69:'A4', 72:'C5'}
pitches, counts = np.unique(y[:,0], return_counts=True)
axes[0,0].bar([pitch_map[int(p)] for p in pitches], counts, color='steelblue', edgecolor='black')
axes[0,0].set_title('Pitch Distribution')
axes[0,0].set_xlabel('Note')
axes[0,0].set_ylabel('Samples')
for i, v in enumerate(counts):
    axes[0,0].text(i, v+1, str(v), ha='center', fontsize=9)

# 2. Phan bo Chord
chord_map = {0:'None', 1:'Major', 2:'Minor', 3:'Dominant'}
chords, ccounts = np.unique(y[:,3], return_counts=True)
colors = ['gray','#2ecc71','#e74c3c','#3498db']
axes[0,1].bar([chord_map[int(c)] for c in chords], ccounts,
              color=[colors[int(c)] for c in chords], edgecolor='black')
axes[0,1].set_title('Chord Type Distribution')
axes[0,1].set_ylabel('Samples')
for i, v in enumerate(ccounts):
    axes[0,1].text(i, v+1, str(v), ha='center', fontsize=9)

# 3. Phan bo Velocity
axes[0,2].hist(y[:,1], bins=20, color='coral', edgecolor='black')
axes[0,2].set_title('Velocity Distribution')
axes[0,2].set_xlabel('Velocity (40-127)')
axes[0,2].set_ylabel('Samples')
axes[0,2].axvline(y[:,1].mean(), color='red', linestyle='--', label=f'Mean={y[:,1].mean():.0f}')
axes[0,2].legend()

# 4. Phan bo Tempo
axes[1,0].hist(y[:,2], bins=20, color='mediumpurple', edgecolor='black')
axes[1,0].set_title('Tempo Distribution')
axes[1,0].set_xlabel('BPM (80-160)')
axes[1,0].set_ylabel('Samples')
axes[1,0].axvline(y[:,2].mean(), color='red', linestyle='--', label=f'Mean={y[:,2].mean():.0f}')
axes[1,0].legend()

# 5. Trang thai tay
hand_labels = ['Only\nRight', 'Only\nLeft', 'Both\nHands', 'No\nHands']
rp, lp = y[:,4], y[:,5]
hand_counts = [
    ((rp==1)&(lp==0)).sum(),
    ((rp==0)&(lp==1)).sum(),
    ((rp==1)&(lp==1)).sum(),
    ((rp==0)&(lp==0)).sum()
]
axes[1,1].bar(hand_labels, hand_counts, color=['#f39c12','#1abc9c','#9b59b6','#95a5a6'], edgecolor='black')
axes[1,1].set_title('Hand Presence Distribution')
axes[1,1].set_ylabel('Samples')
for i, v in enumerate(hand_counts):
    axes[1,1].text(i, v+1, str(v), ha='center', fontsize=9)

# 6. Vi du 1 sequence
sample_seq = X[50, :, :3]  # 30 frames, chi lay 3 features dau
for i in range(3):
    axes[1,2].plot(sample_seq[:, i], label=f'lm0_{["x","y","z"][i]}')
axes[1,2].set_title('Sample Sequence (Right Hand Wrist)')
axes[1,2].set_xlabel('Frame (0-29)')
axes[1,2].set_ylabel('Normalized Value')
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dataset_analysis.png')

## 2. Model & Training

In [ ]:
PITCH_MIN, PITCH_MAX = 60, 72
VEL_MIN,   VEL_MAX   = 40, 127
TEMPO_MIN, TEMPO_MAX = 80, 160

y_reg = y.copy().astype(np.float32)
y_reg[:, 0] = (y[:, 0] - PITCH_MIN) / (PITCH_MAX - PITCH_MIN)
y_reg[:, 1] = (y[:, 1] - VEL_MIN)   / (VEL_MAX - VEL_MIN)
y_reg[:, 2] = (y[:, 2] - TEMPO_MIN) / (TEMPO_MAX - TEMPO_MIN)

class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

dataset = GestureDataset(X, y_reg)
n_train = int(0.8 * len(dataset))
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
print(f'Train: {n_train} | Val: {n_val}')

In [ ]:
class GestureTransformer(nn.Module):
    def __init__(self, input_dim=126, d_model=128, nhead=4, num_layers=3, output_dim=6):
        super().__init__()
        self.embed   = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Embedding(100, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, output_dim)

    def forward(self, x):
        B, T, _ = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_enc(pos)
        x = self.transformer(x)
        return self.head(x.mean(dim=1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = GestureTransformer().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Device: {device} | Params: {total_params:,}')

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)
mse = nn.MSELoss()
bce = nn.BCEWithLogitsLoss()

def compute_loss(pred, target):
    loss_reg   = mse(pred[:, :3], target[:, :3])
    loss_cls   = bce(pred[:, 4:], target[:, 4:])
    loss_chord = mse(pred[:, 3:4], target[:, 3:4] / 3.0)
    return loss_reg + 0.5 * loss_cls + 0.3 * loss_chord

EPOCHS = 60
train_losses, val_losses, lr_history = [], [], []

for epoch in range(EPOCHS):
    model.train()
    t_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = compute_loss(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            v_loss += compute_loss(model(xb), yb).item()

    scheduler.step()
    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss / len(val_loader))
    lr_history.append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f}')

print('Done!')

## 3. Bieu do Training & Danh gia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('GestuRhythm - Training Results', fontsize=14, fontweight='bold')

# 1. Loss curve
axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Learning rate schedule
axes[1].plot(lr_history, color='green')
axes[1].set_title('Learning Rate Schedule (Cosine Annealing)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('LR')
axes[1].grid(True, alpha=0.3)

# 3. Pred vs True tren val set
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        pred = model(xb.to(device)).cpu().numpy()
        all_pred.append(pred)
        all_true.append(yb.numpy())
all_pred = np.vstack(all_pred)
all_true = np.vstack(all_true)

# Denormalize pitch de ve bieu do
pred_pitch = all_pred[:,0] * (PITCH_MAX - PITCH_MIN) + PITCH_MIN
true_pitch = all_true[:,0] * (PITCH_MAX - PITCH_MIN) + PITCH_MIN
axes[2].scatter(true_pitch, pred_pitch, alpha=0.5, color='mediumpurple', s=20)
axes[2].plot([60,72],[60,72], 'r--', label='Perfect')
axes[2].set_title('Pitch: Predicted vs True')
axes[2].set_xlabel('True Pitch')
axes[2].set_ylabel('Predicted Pitch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_results.png')

In [ ]:
# Confusion matrix cho Chord classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

pred_chord = np.round(all_pred[:,3] * 3).clip(0,3).astype(int)
true_chord = all_true[:,3].astype(int)

cm = confusion_matrix(true_chord, pred_chord, labels=[0,1,2,3])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('GestuRhythm - Classification Results', fontsize=14, fontweight='bold')

disp = ConfusionMatrixDisplay(cm, display_labels=['None','Major','Minor','Dom'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Chord Type - Confusion Matrix')

# Confusion matrix cho Hand presence
pred_rp = (all_pred[:,4] > 0).astype(int)
true_rp = all_true[:,4].astype(int)
cm2 = confusion_matrix(true_rp, pred_rp)
disp2 = ConfusionMatrixDisplay(cm2, display_labels=['No Hand','Hand'])
disp2.plot(ax=axes[1], colorbar=False)
axes[1].set_title('Right Hand Presence - Confusion Matrix')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Chord Classification Report:')
print(classification_report(true_chord, pred_chord,
      target_names=['None','Major','Minor','Dominant']))

In [ ]:
# MAE cho regression outputs
pred_vel   = all_pred[:,1] * (VEL_MAX - VEL_MIN) + VEL_MIN
true_vel   = all_true[:,1] * (VEL_MAX - VEL_MIN) + VEL_MIN
pred_tempo = all_pred[:,2] * (TEMPO_MAX - TEMPO_MIN) + TEMPO_MIN
true_tempo = all_true[:,2] * (TEMPO_MAX - TEMPO_MIN) + TEMPO_MIN

mae_pitch = np.abs(pred_pitch - true_pitch).mean()
mae_vel   = np.abs(pred_vel   - true_vel).mean()
mae_tempo = np.abs(pred_tempo - true_tempo).mean()

print('=== FINAL METRICS ===')
print(f'MAE Pitch    : {mae_pitch:.2f} semitones')
print(f'MAE Velocity : {mae_vel:.2f} / 127')
print(f'MAE Tempo    : {mae_tempo:.2f} BPM')
print(f'Final Val Loss: {val_losses[-1]:.4f}')

In [ ]:
torch.save({
    'model_state': model.state_dict(),
    'config': {'input_dim':126,'d_model':128,'nhead':4,'num_layers':3,'output_dim':6},
    'label_config': {
        'pitch_min':PITCH_MIN,'pitch_max':PITCH_MAX,
        'vel_min':VEL_MIN,'vel_max':VEL_MAX,
        'tempo_min':TEMPO_MIN,'tempo_max':TEMPO_MAX,
        'pitch_map':[60,62,65,69,72],
        'chord_map':{0:'none',1:'major',2:'minor',3:'dominant'}
    }
}, 'gesture_transformer.pt')
print('Saved: gesture_transformer.pt')